# Robust Pipelines, Deployment, and Course Review

> **Advanced · Engineering capstone**


## Why this matters

A notebook proves an idea; a deployable pipeline has a contract, validation, tests, observability, and explicit failure behavior. This capstone consolidates those habits.

**Where it appears:** Reusable Python packages, REST APIs, batch jobs, internal tools, reproducible experiments, and a final end-to-end project.


## Learning Objectives

- Package an OpenCV pipeline as a reusable, testable module (not notebook-only code)
- Wrap a pipeline behind a minimal REST API for real deployment
- Apply basic input validation and error handling expected in production
- Consolidate the recurring engineering patterns used throughout the series
- Apply a checklist-driven code review to a deliberately flawed pipeline
- Know where to go next based on your specific application area


## Prerequisites

At least one complete processing pipeline from this course; 24 Performance and Hardware Acceleration is recommended

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

pipeline classes, input validation, FastAPI integration, health checks, logging, tests, profiling

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Deployment

Deployment means the difference between "runs in my notebook" and "runs
reliably for other callers, including malformed input, at 3am, without a
human watching." That requires: packaging pipeline logic as importable
functions/classes with clear contracts, validating input rather than
trusting it, returning structured errors instead of raising raw
exceptions to callers, and having at least a minimal automated test.


### Best Practices and Final Review

This closing notebook does not introduce new OpenCV functionality --
instead it consolidates the patterns that recurred across all 39 previous
notebooks (modular functions, defensive I/O, vectorization, resource
cleanup, validated deployment) into a single checklist, then applies that
checklist to find and fix real problems in a deliberately flawed example
pipeline -- the same kind of review this whole rewritten series is meant
to model.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Deployment


### 1. Packaging a pipeline as a class with a clear contract

Wrap a full pipeline (using patterns from every earlier notebook) behind a single class with an explicit `process()` method and a documented input/output contract.


In [ ]:
import cv2
import numpy as np
from dataclasses import dataclass
from cv_utils import load_real_image, get_real_data


@dataclass
class DetectionResult:
    shape_count: int
    boxes: list


class ShapeDetectionService:
    """A minimal, deployable service wrapping the notebook 14 shape-classification pipeline.
    Contract: process() takes a BGR uint8 numpy array, returns a DetectionResult, or raises
    ValueError for invalid input (never a raw/unexplained exception)."""

    def __init__(self, min_area: int = 200):
        self.min_area = min_area

    def process(self, image: np.ndarray) -> DetectionResult:
        self._validate(image)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        significant = [c for c in contours if cv2.contourArea(c) >= self.min_area]
        boxes = [cv2.boundingRect(c) for c in significant]
        return DetectionResult(shape_count=len(boxes), boxes=boxes)

    @staticmethod
    def _validate(image) -> None:
        if not isinstance(image, np.ndarray):
            raise ValueError(f"Expected a numpy array, got {type(image)}")
        if image.ndim != 3 or image.shape[2] != 3:
            raise ValueError(f"Expected a 3-channel BGR image, got shape {image.shape}")
        if image.dtype != np.uint8:
            raise ValueError(f"Expected dtype uint8, got {image.dtype}")


service = ShapeDetectionService()
result = service.process(load_real_image("images/landscapes", "mountain.jpg"))
print(result)

### 2. Validating and rejecting bad input explicitly

A deployed service must reject malformed input with a clear message instead of crashing deep inside OpenCV -- test the validation path directly.


In [ ]:
bad_inputs = [
    ("wrong type", [1, 2, 3]),
    ("wrong dtype", np.zeros((100, 100, 3), dtype=np.float32)),
    ("wrong channels", np.zeros((100, 100), dtype=np.uint8)),
]

for label, bad_input in bad_inputs:
    try:
        service.process(bad_input)
        print(f"{label}: unexpectedly succeeded (should have raised)")
    except ValueError as e:
        print(f"{label}: correctly rejected -> {e}")

### 3. A minimal REST API wrapper

Sketch (not run, since it needs a live server) a minimal FastAPI wrapper around the service -- the standard shape for exposing an OpenCV pipeline as a network endpoint.


In [ ]:
try:
    from fastapi import FastAPI, UploadFile, HTTPException
    app = FastAPI()
    service = ShapeDetectionService()

    @app.post("/detect")
    async def detect(file: UploadFile):
        raw_bytes = await file.read()
        array = np.frombuffer(raw_bytes, dtype=np.uint8)
        image = cv2.imdecode(array, cv2.IMREAD_COLOR)
        if image is None:
            raise HTTPException(status_code=400, detail="Could not decode uploaded file as an image")
        try:
            result = service.process(image)
        except ValueError as e:
            raise HTTPException(status_code=422, detail=str(e))
        return {"shape_count": result.shape_count, "boxes": result.boxes}
    print("FastAPI app initialized. Run with: uvicorn app:app --host 0.0.0.0 --port 8000")
except ImportError:
    print("FastAPI is not installed. To run the API, install fastapi and uvicorn.")


## Part 2: Best Practices and Final Review


### 1. The recurring patterns, consolidated

A single reference list of the engineering habits introduced piecemeal across the series -- worth keeping open while writing new OpenCV code.


In [ ]:
BEST_PRACTICES_CHECKLIST = {
    "Structure": [
        "Pipeline steps are functions with clear inputs/outputs, not flat top-level script code (NB 01, 03).",
        "Shared logic (display, synthetic data, timing) lives in one reusable module, not copy-pasted (NB 01, this series' cv_utils.py).",
    ],
    "Correctness": [
        "Images are treated as NumPy arrays with known shape/dtype; arithmetic is cast/clipped explicitly (NB 02).",
        "BGR vs RGB is converted deliberately at every library boundary (NB 03, 05, 33).",
        "cv2.imread/VideoCapture failures are checked, not assumed to succeed (NB 01, 05, 20).",
    ],
    "Robustness": [
        "Thresholding/detection parameters are derived or justified, not guessed (NB 12, 13).",
        "Detections are filtered by confidence/area and de-duplicated with NMS where relevant (NB 14, 19, 24, 35).",
        "Model preprocessing (scale, mean, channel order) matches the specific model's documented contract (NB 34, 36).",
    ],
    "Resources & Performance": [
        "VideoCapture/VideoWriter/network resources are released in try/finally (NB 20).",
        "Performance work starts with profiling, not guessing (NB 37).",
        "GPU code paths check availability and fall back to CPU (NB 38).",
    ],
    "Deployment": [
        "Public entry points validate input and raise clear, structured errors (NB 39).",
        "There is at least a minimal automated test for core logic (NB 39).",
    ],
}

for category, items in BEST_PRACTICES_CHECKLIST.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  - {item}")

### 2. Robust Image Processing Pipeline

A production-ready pipeline applying the checklist: I/O validation, contours filtered by minimum area, function structure, and no silent failure paths.

In [ ]:
from pathlib import Path
import cv2
import numpy as np


def find_and_box_shapes(image_path: str, min_area: int = 100) -> np.ndarray:
    """Detect shape contours in `image_path` and return an annotated copy.
    Raises FileNotFoundError if the image can't be loaded (Violation: original had no check).
    """
    image = cv2.imread(str(image_path))
    if image is None:  # FIX: check imread result
        raise FileNotFoundError(f"Could not load image: {image_path}")

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    significant = [
        c for c in contours if cv2.contourArea(c) >= min_area
    ]  # FIX: filter noise

    annotated = image.copy()  # FIX: don't mutate in place blindly
    for c in significant:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return annotated

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Deployment: Latency Checker and Healthcheck Middleware

When deploying models as microservices, endpoints must respond within latency budget envelopes. Here, we build a timing middleware decorator to log execution latencies.


In [ ]:
import time
import functools


def measure_latency_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - t0) * 1000
        print(f"[API Monitor] {func.__name__} latency: {elapsed:.2f} ms")
        return result

    return wrapper


@measure_latency_decorator
def process_frame_api(frame: np.ndarray):
    # Mock model inference
    time.sleep(0.015)  # Simulate processing delay
    return {"status": "SUCCESS", "count": 2}


dummy = np.zeros((100, 100, 3), dtype=np.uint8)
res = process_frame_api(dummy)

### Mini Project — Best Practices and Final Review: Static Code Linter for OpenCV Codebases

To enforce standards automatically, we build a static text linter that scans Python files for direct loops or unsafe load calls.


In [ ]:
# Mock source code containing violations
mock_source = """
import cv2
img = cv2.imread("photo.jpg") # Violation: unsafe read
h, w = img.shape[:2]
for y in range(h): # Violation: nested loops
    for x in range(w):
        img[y, x] = img[y, x] + 10
"""


def lint_opencv_code(source: str) -> list[str]:
    violations = []
    lines = source.split("\n")
    for idx, line in enumerate(lines):
        if "cv2.imread(" in line and "safe_imread" not in line:
            violations.append(
                f"Line {idx + 1}: Unsafe cv2.imread call. Use safe_imread instead."
            )
        if "for y in range" in line or "for x in range" in line:
            violations.append(
                f"Line {idx + 1}: Python loops detected. Use vectorized NumPy expressions."
            )
    return violations


reports = lint_opencv_code(mock_source)
print("Static Code Linter reports:")
for r in reports:
    print(f"  [VIOLATION] {r}")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Deployment
1. Write 3 `assert`-based unit tests for `ShapeDetectionService.process` covering: a normal case, an empty (all-white) image, and an oversized image.
2. Add request size/dimension limits to the sketched API to reject excessively large uploaded images before processing.
3. Containerize the service conceptually: list (in markdown) the contents of a minimal Dockerfile that would run this API.

Use the empty cell below to work through them.


#### Solutions — Deployment

**Solution 1: Unit tests for ShapeDetectionService**

1. **Normal Image**: Verify output dict contains expected keys ('shapes') and returns status 200.
2. **All-White Image**: Verify processing handles flat empty regions without throwing exceptions.
3. **Oversized Image**: Verify system triggers a validation error if image exceeds allowed limits.

In [ ]:
def validate_incoming_frame(image: np.ndarray, max_w: int = 1920, max_h: int = 1080) -> None:
    """Reject input images exceeding dimension limits before parsing."""
    h, w = image.shape[:2]
    if w > max_w or h > max_h:
        raise ValueError(
            f"Image dimensions ({w}x{h}) exceed maximum allowed scale ({max_w}x{max_h})"
        )
    print("Incoming frame validation: PASSED")


**Solution 3: Minimal Dockerfile configuration**

```dockerfile
FROM python:3.10-slim
RUN apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["python", "main.py"]
```

### Exercises — Best Practices and Final Review
1. Pick any 3 notebooks from this series and write a one-paragraph summary of the single most important idea in each.
2. Take a piece of your own existing OpenCV code and run it against the `BEST_PRACTICES_CHECKLIST` -- list every violation you find.
3. Based on your intended application (e.g. robotics, web app, mobile, research), identify which 5 notebooks in this series are most directly relevant and re-read those first when starting a real project.

Use the empty cell below to work through them.


#### Solutions — Best Practices and Final Review

**Solution 1: Notebook Summaries**

1. **02 NumPy for CV**: OpenCV images are numpy arrays. Vectorized slice indexing (slicing) replaces expensive pixel loops and prevents memory copy operations.
2. **07 Color Spaces**: Decoupling luminance from chromatic variables (HSV) enables color thresholding that remains robust to shading and light source changes.
3. **37 Performance Optimization**: Profiling must precede optimization. Vectorization and multithreading deliver massive computational speedups.

**Solution 2: Violation Checklist Audit**

- **Violation 1**: Using `cv2.imread` directly without error checks -> fix with `safe_imread`.
- **Violation 2**: Nested loops to adjust brightness -> fix with `np.clip(img + delta, 0, 255)`.
- **Violation 3**: String path concatenation -> fix with `pathlib.Path`.

**Solution 3: Relevance Mapping by Domain**

Depending on the application domain, the most relevant notebooks differ:
- **Web/REST API Deployment**: 05 Image IO, 34 DNN Module, 36 ONNX, 37 Perf, 39 Deployment.
- **Robotics/Autonomous Systems**: 09 Geo, 22 Tracking, 29 Calibration, 32 Pose, 38 GPU.
- **Classical Document OCR**: 06 ROI, 13 Thresholding, 14 Morphology, 15 Edges, 27 OCR.

## Summary

You can move a notebook prototype into a small, testable service or package and apply a practical checklist to future vision work.

- **Best Practices:** Define input/output contracts, test normal and failure paths, pin dependencies, log model/data versions, monitor latency, and document privacy/security constraints.
- **Common Pitfalls:** Deploying notebook state, accepting unbounded uploads, hiding errors, and treating an API demo as production readiness.